In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/energy.db")
print("✓ Connected to energy.db")

print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
).to_string(index=False))

✓ Connected to energy.db
          name
  daily_demand
monthly_demand


In [2]:
# ── Query 1: Business Summary KPIs ──────────────────────────

query = """
SELECT
    COUNT(*)                                    AS total_days,
    ROUND(SUM(demand_mwh), 1)                  AS total_demand_mwh,
    ROUND(AVG(demand_mwh), 1)                  AS avg_daily_demand,
    ROUND(MAX(demand_mwh), 1)                  AS peak_demand,
    ROUND(MIN(demand_mwh), 1)                  AS min_demand,
    SUM(is_spike)                               AS total_spike_days
FROM daily_demand
"""
pd.read_sql(query, conn)

,total_days,total_demand_mwh,avg_daily_demand,peak_demand,min_demand,total_spike_days
0,1096,1328272.5,1211.9,2170.3,643.6,19


In [3]:
#Annual demand growth

# ── Query 2: Year-over-Year Demand Growth ────────────────────

query = """
SELECT
    year,
    COUNT(*)                                    AS days,
    ROUND(SUM(demand_mwh), 1)                  AS total_demand,
    ROUND(AVG(demand_mwh), 1)                  AS avg_daily_demand,
    ROUND(MAX(demand_mwh), 1)                  AS peak_demand,
    ROUND(SUM(demand_mwh) * 100.0 /
        SUM(SUM(demand_mwh)) OVER (), 1)       AS pct_of_total
FROM daily_demand
GROUP BY year
ORDER BY year
"""
pd.read_sql(query, conn)

,year,days,total_demand,avg_daily_demand,peak_demand,pct_of_total
0,2022,365,427690.2,1171.8,1783.4,32.2
1,2023,365,446849.9,1224.2,2043.9,33.6
2,2024,366,453732.4,1239.7,2170.3,34.2


In [4]:
# ── Query 3: Demand by Season ────────────────────────────────

query = """
SELECT
    season,
    COUNT(*)                                    AS days,
    ROUND(AVG(demand_mwh), 1)                  AS avg_daily_demand,
    ROUND(MAX(demand_mwh), 1)                  AS peak_demand,
    ROUND(MIN(demand_mwh), 1)                  AS min_demand,
    ROUND(AVG(temperature_f), 1)               AS avg_temperature
FROM daily_demand
GROUP BY season
ORDER BY avg_daily_demand DESC
"""
pd.read_sql(query, conn)

,season,days,avg_daily_demand,peak_demand,min_demand,avg_temperature
0,Summer,276,1373.0,2170.3,878.6,86.7
1,Winter,271,1304.8,2013.7,876.1,37.3
2,Fall,273,1123.6,1675.6,693.1,64.0
3,Spring,276,1047.1,1504.6,643.6,62.0


In [5]:
# ── Query 4: Demand by Day of Week ───────────────────────────
# Shows weekday vs weekend pattern

query = """
SELECT
    day_of_week,
    COUNT(*)                                    AS days,
    ROUND(AVG(demand_mwh), 1)                  AS avg_demand,
    ROUND(MAX(demand_mwh), 1)                  AS peak_demand,
    ROUND(AVG(demand_mwh) * 100.0 /
        AVG(AVG(demand_mwh)) OVER (), 1)       AS pct_vs_average
FROM daily_demand
GROUP BY day_of_week
ORDER BY avg_demand DESC
"""
pd.read_sql(query, conn)

,day_of_week,days,avg_demand,peak_demand,pct_vs_average
0,Thursday,156,1325.6,2043.9,109.4
1,Tuesday,157,1322.6,1835.1,109.1
2,Wednesday,156,1309.2,1811.6,108.0
3,Monday,157,1282.1,2170.3,105.8
4,Friday,156,1274.8,1737.8,105.2
5,Saturday,157,1000.1,1369.3,82.5
6,Sunday,157,970.8,1396.7,80.1


In [6]:
# ── Query 5: Monthly Demand Pattern ─────────────────────────
# Shows seasonal cycle across all years

query = """
SELECT
    month,
    ROUND(AVG(total_demand), 1)                AS avg_monthly_demand,
    ROUND(AVG(avg_daily), 1)                   AS avg_daily_demand,
    ROUND(AVG(avg_temp), 1)                    AS avg_temperature,
    ROUND(MAX(peak_demand), 1)                 AS highest_peak
FROM monthly_demand
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,avg_monthly_demand,avg_daily_demand,avg_temperature,highest_peak
0,1,41365.0,1334.3,35.2,1739.6
1,2,36322.9,1281.7,37.8,2013.7
2,3,34848.7,1124.2,51.4,1412.7
3,4,30635.7,1021.2,62.9,1393.8
4,5,30846.1,995.0,71.8,1504.6
5,6,37098.4,1236.6,81.7,1596.1
6,7,45774.3,1476.6,90.1,2170.3
7,8,43439.2,1401.3,88.2,2043.9
8,9,35603.9,1186.8,78.2,1675.6
9,10,32337.7,1043.1,63.8,1409.5


In [7]:
# ── Query 6: Temperature vs Demand ──────────────────────────
# Shows relationship between temperature and energy use

query = """
SELECT
    CASE
        WHEN temperature_f < 40  THEN 'Below 40F'
        WHEN temperature_f < 55  THEN '40-55F'
        WHEN temperature_f < 70  THEN '55-70F'
        WHEN temperature_f < 80  THEN '70-80F'
        WHEN temperature_f < 90  THEN '80-90F'
        ELSE 'Above 90F'
    END                                         AS temp_range,
    COUNT(*)                                    AS days,
    ROUND(AVG(demand_mwh), 1)                  AS avg_demand,
    ROUND(MIN(demand_mwh), 1)                  AS min_demand,
    ROUND(MAX(demand_mwh), 1)                  AS max_demand
FROM daily_demand
GROUP BY temp_range
ORDER BY avg_demand DESC
"""
pd.read_sql(query, conn)

,temp_range,days,avg_demand,min_demand,max_demand
0,Above 90F,91,1450.1,925.5,2170.3
1,80-90F,186,1317.3,795.8,2043.9
2,Below 40F,190,1305.1,876.1,2013.7
3,40-55F,229,1185.5,693.1,1670.9
4,70-80F,159,1113.4,643.6,1707.1
5,55-70F,241,1057.3,655.0,1675.6


In [8]:
# ── Query 7: Spike Day Analysis ──────────────────────────────
# Identifies anomaly days — extreme demand events

query = """
SELECT
    date,
    season,
    day_of_week,
    temperature_f,
    demand_mwh,
    ROUND(demand_mwh - AVG(demand_mwh) OVER (
        PARTITION BY month
    ), 1)                                       AS demand_vs_monthly_avg
FROM daily_demand
WHERE is_spike = 1
ORDER BY demand_mwh DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,date,season,day_of_week,temperature_f,demand_mwh,demand_vs_monthly_avg
0,2024-07-29 00:00:00,Summer,Monday,98.1,2170.3,0.0
1,2023-08-10 00:00:00,Summer,Thursday,87.7,2043.9,0.0
2,2023-02-23 00:00:00,Winter,Thursday,33.0,2013.7,0.0
3,2022-09-23 00:00:00,Fall,Friday,69.7,1675.6,230.4
4,2022-12-28 00:00:00,Winter,Wednesday,41.7,1648.4,0.0
5,2022-01-20 00:00:00,Winter,Thursday,36.0,1628.4,143.5
6,2023-11-02 00:00:00,Fall,Thursday,50.0,1545.5,0.0
7,2023-05-23 00:00:00,Spring,Tuesday,73.0,1504.6,158.3
8,2024-05-17 00:00:00,Spring,Friday,63.9,1454.4,108.1
9,2022-10-13 00:00:00,Fall,Thursday,72.4,1409.5,143.2


In [9]:
# ── Query 7: Spike Day Analysis ──────────────────────────────
# Identifies anomaly days — extreme demand events

query = """
SELECT
    date,
    season,
    day_of_week,
    temperature_f,
    demand_mwh,
    ROUND(demand_mwh - AVG(demand_mwh) OVER (
        PARTITION BY month
    ), 1)                                       AS demand_vs_monthly_avg
FROM daily_demand
WHERE is_spike = 1
ORDER BY demand_mwh DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,date,season,day_of_week,temperature_f,demand_mwh,demand_vs_monthly_avg
0,2024-07-29 00:00:00,Summer,Monday,98.1,2170.3,0.0
1,2023-08-10 00:00:00,Summer,Thursday,87.7,2043.9,0.0
2,2023-02-23 00:00:00,Winter,Thursday,33.0,2013.7,0.0
3,2022-09-23 00:00:00,Fall,Friday,69.7,1675.6,230.4
4,2022-12-28 00:00:00,Winter,Wednesday,41.7,1648.4,0.0
5,2022-01-20 00:00:00,Winter,Thursday,36.0,1628.4,143.5
6,2023-11-02 00:00:00,Fall,Thursday,50.0,1545.5,0.0
7,2023-05-23 00:00:00,Spring,Tuesday,73.0,1504.6,158.3
8,2024-05-17 00:00:00,Spring,Friday,63.9,1454.4,108.1
9,2022-10-13 00:00:00,Fall,Thursday,72.4,1409.5,143.2


In [10]:
# ── Query 8: YoY Monthly Comparison ─────────────────────────
# Compares each month in 2024 vs 2023

query = """
WITH y2023 AS (
    SELECT month,
           ROUND(AVG(avg_daily), 1)     AS avg_daily_2023
    FROM monthly_demand
    WHERE year = 2023
    GROUP BY month
),
y2024 AS (
    SELECT month,
           ROUND(AVG(avg_daily), 1)     AS avg_daily_2024
    FROM monthly_demand
    WHERE year = 2024
    GROUP BY month
)
SELECT
    y2024.month,
    y2023.avg_daily_2023,
    y2024.avg_daily_2024,
    ROUND(y2024.avg_daily_2024 -
          y2023.avg_daily_2023, 1)      AS change_mwh,
    ROUND((y2024.avg_daily_2024 -
           y2023.avg_daily_2023) * 100.0 /
           y2023.avg_daily_2023, 1)     AS growth_pct
FROM y2024
JOIN y2023 ON y2024.month = y2023.month
ORDER BY y2024.month
"""
pd.read_sql(query, conn)

,month,avg_daily_2023,avg_daily_2024,change_mwh,growth_pct
0,1,1344.0,1384.8,40.8,3.0
1,2,1312.2,1307.6,-4.6,-0.4
2,3,1144.2,1134.9,-9.3,-0.8
3,4,1012.8,1061.3,48.5,4.8
4,5,1007.1,1018.7,11.6,1.2
5,6,1258.1,1228.8,-29.3,-2.3
6,7,1496.3,1527.1,30.8,2.1
7,8,1405.0,1440.9,35.9,2.6
8,9,1195.2,1209.9,14.7,1.2
9,10,1029.3,1084.3,55.0,5.3


In [11]:
# ── Query 9: 7-Day Rolling Average ───────────────────────────
# Smooths out daily noise — feeds directly into forecasting
# Shows window function skills with ROWS BETWEEN

query = """
SELECT
    date,
    demand_mwh,
    ROUND(AVG(demand_mwh) OVER (
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 1)                               AS rolling_7day_avg,
    ROUND(AVG(demand_mwh) OVER (
        ORDER BY date
        ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ), 1)                               AS rolling_30day_avg
FROM daily_demand
ORDER BY date
"""
pd.read_sql(query, conn).tail(10)

,date,demand_mwh,rolling_7day_avg,rolling_30day_avg
1086,2024-12-22 00:00:00,1047.0,1355.3,1275.4
1087,2024-12-23 00:00:00,1345.2,1360.0,1286.0
1088,2024-12-24 00:00:00,1279.0,1324.8,1297.9
1089,2024-12-25 00:00:00,1705.5,1332.4,1321.4
1090,2024-12-26 00:00:00,1432.8,1342.8,1318.2
1091,2024-12-27 00:00:00,1395.3,1343.8,1322.1
1092,2024-12-28 00:00:00,917.1,1303.1,1310.2
1093,2024-12-29 00:00:00,952.6,1289.6,1304.7
1094,2024-12-30 00:00:00,1392.8,1296.4,1316.8
1095,2024-12-31 00:00:00,1280.9,1296.7,1322.5


In [12]:
# ── Query 10: Peak Demand Days by Season ─────────────────────
# Ranks highest demand days within each season

query = """
SELECT
    date,
    season,
    year,
    temperature_f,
    demand_mwh,
    RANK() OVER (
        PARTITION BY season
        ORDER BY demand_mwh DESC
    )                                   AS rank_in_season,
    RANK() OVER (
        ORDER BY demand_mwh DESC
    )                                   AS overall_rank
FROM daily_demand
QUALIFY RANK() OVER (
    PARTITION BY season
    ORDER BY demand_mwh DESC
) <= 5
ORDER BY season, rank_in_season
"""

# Note: QUALIFY is not supported in SQLite
# Use this version instead:
query = """
WITH ranked AS (
    SELECT
        date,
        season,
        year,
        temperature_f,
        demand_mwh,
        RANK() OVER (
            PARTITION BY season
            ORDER BY demand_mwh DESC
        )                               AS rank_in_season,
        RANK() OVER (
            ORDER BY demand_mwh DESC
        )                               AS overall_rank
    FROM daily_demand
)
SELECT * FROM ranked
WHERE rank_in_season <= 5
ORDER BY season, rank_in_season
"""
pd.read_sql(query, conn)

,date,season,year,temperature_f,demand_mwh,rank_in_season,overall_rank
0,2022-09-23 00:00:00,Fall,2022,69.7,1675.6,1,28
1,2023-11-02 00:00:00,Fall,2023,50.0,1545.5,2,98
2,2024-11-26 00:00:00,Fall,2024,53.9,1528.7,3,107
3,2023-09-21 00:00:00,Fall,2023,80.1,1456.6,4,158
4,2023-11-23 00:00:00,Fall,2023,47.0,1445.3,5,172
5,2023-05-23 00:00:00,Spring,2023,73.0,1504.6,1,119
6,2024-05-17 00:00:00,Spring,2024,63.9,1454.4,2,161
7,2023-03-17 00:00:00,Spring,2023,47.3,1412.7,3,219
8,2023-04-17 00:00:00,Spring,2023,61.5,1393.8,4,243
9,2023-03-08 00:00:00,Spring,2023,59.0,1368.5,5,284


In [13]:
sql_queries = """
-- ============================================================
-- ENERGY DEMAND FORECASTING — SQL Queries
-- Database: data/energy.db
-- Tables: daily_demand, monthly_demand
-- ============================================================

-- 1. BUSINESS SUMMARY
SELECT
    COUNT(*)                            AS total_days,
    ROUND(SUM(demand_mwh), 1)          AS total_demand_mwh,
    ROUND(AVG(demand_mwh), 1)          AS avg_daily_demand,
    ROUND(MAX(demand_mwh), 1)          AS peak_demand
FROM daily_demand;

-- 2. YEAR OVER YEAR GROWTH
SELECT
    year,
    ROUND(SUM(demand_mwh), 1)          AS total_demand,
    ROUND(AVG(demand_mwh), 1)          AS avg_daily_demand,
    ROUND(SUM(demand_mwh) * 100.0 /
        SUM(SUM(demand_mwh)) OVER (), 1) AS pct_of_total
FROM daily_demand
GROUP BY year
ORDER BY year;

-- 3. DEMAND BY SEASON
SELECT
    season,
    ROUND(AVG(demand_mwh), 1)          AS avg_daily_demand,
    ROUND(AVG(temperature_f), 1)       AS avg_temperature
FROM daily_demand
GROUP BY season
ORDER BY avg_daily_demand DESC;

-- 4. 7-DAY ROLLING AVERAGE
SELECT
    date,
    demand_mwh,
    ROUND(AVG(demand_mwh) OVER (
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 1)                               AS rolling_7day_avg
FROM daily_demand
ORDER BY date;

-- 5. PEAK DEMAND DAYS BY SEASON
WITH ranked AS (
    SELECT date, season, year,
           temperature_f, demand_mwh,
           RANK() OVER (
               PARTITION BY season
               ORDER BY demand_mwh DESC
           )                            AS rank_in_season
    FROM daily_demand
)
SELECT * FROM ranked
WHERE rank_in_season <= 5
ORDER BY season, rank_in_season;
"""

with open("../sql/energy_queries.sql", "w") as f:
    f.write(sql_queries)

conn.close()
print("✓ SQL file saved to sql/energy_queries.sql")
print("✓ Connection closed")

✓ SQL file saved to sql/energy_queries.sql
✓ Connection closed
